In [1]:
# Install required packages
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import copy
import pandas as pd
from datetime import date

from scipy.optimize import minimize, check_grad, approx_fprime
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Import pyomo environment and setup gurobi solver
import pyomo.environ as pyo
from pyomo.opt import SolverStatus, TerminationCondition
import gurobipy 
solver = pyo.SolverFactory("gurobi_direct")

import time

In [2]:
# Global variables
nr_items = 25
smple_sz = 1000
rsmpl_sz = 10
epsilon  = 10e-4
today = date.today()

# Implied variables
nr_trgts = nr_items + 1
nr_ftres = 2*nr_items
nr_ftres_intrcpt = nr_ftres + 1

In [7]:
# Set seeds
seed_lst = [*range(42, 42+rsmpl_sz)] 
np.random.seed(seed_lst[0])

In [3]:
# Source types: https://github.com/likr/kplib
# Type discription source: https://di.ku.dk/forskning/Publikationer/tekniske_rapporter/tekniske-rapporter-2003/03-08.pdf

ks_types = ['00Uncorrelated', '01WeaklyCorrelated', '02StronglyCorrelated', '03InverseStronglyCorrelated']
tp_insts = ['/s00'+str(i)+'.kp' for i in range(10)]

nr_typs  = len(ks_types)
nr_inst  = len(tp_insts)
f_list   = []

ins_b    = []
ins_v    = []
ins_w    = [] 

for ks_type in ks_types:
    for tp_inst in tp_insts:
        file_str = 'Instances/' + ks_type + tp_inst
        f = open(file_str, "r")
        f_list.append(f)
        i = 0
        lcl_v = []
        lcl_w = []
        for x in f:
            if i == 2:
                ins_b.append(int(x))
            if i > 3:
                x_split = x.split()
                lcl_v.append(int(x_split[0]))
                lcl_w.append(int(x_split[1]))
            i = i+1

        ins_v.append(lcl_v)
        ins_w.append(lcl_w)

In [4]:
# Continuous LP knapsack
def model_KS_cts(vals, output = 'goal', solver = solver):

    # Define a model
    model = pyo.ConcreteModel('Knapsack continuous model')

    # Declare decision variables
    model.x = pyo.Var(range(int(((len(vals)-1)/2))), domain=pyo.NonNegativeReals, bounds=(0, 1))

    # Declare objective
    model.objective = pyo.Objective(expr = sum(vals[i]*model.x[i] for i in range(int(((len(vals)-1)/2)))),
                                sense = pyo.maximize)

    # Declare constraints
    model.budget = pyo.Constraint(expr = sum(vals[i+int(((len(vals)-1)/2))]*model.x[i] for i in range(int(((len(vals)-1)/2)))) <= vals[-1])

    # Solve
    result = solver.solve(model)

    if output == 'goal':
        return model.objective()

    elif output == 'bounded' or output == 'feasibility':
        return result.solver.termination_condition != TerminationCondition.infeasibleOrUnbounded
    
    elif output == 'decision vector':
        solutions = []
        for i in range(int(((len(vals)-1)/2))):
            solutions.append(pyo.value(model.x[i]))
        return solutions
    
    elif output == 'all':
        solutions = []
        solutions.append(model.objective())
        for i in range(int(((len(vals)-1)/2))):
            solutions.append(pyo.value(model.x[i]))
        return solutions
    
    else:
        raise ValueError("Output not supported for model function")

In [5]:
ins_dict = {}
for i in range(nr_typs):
    print('i',i)
    ins_dict['Type '+str(i)] = {}
    for j in range(nr_inst):
        print('j',j)
        indx = int(i + j)
        print('indc',indx)
        ins_lcl_v = [x/ins_b[indx]*2 for x in ins_v[indx][:nr_items]]
        ins_lcl_w = [x/ins_b[indx]*2 for x in ins_w[indx][:nr_items]]
        print(sum(ins_lcl_w))
        ins_lcl_b = [1.0]
        ins_dict['Type '+str(i)]['Instance '+str(j)] = {}
        ins_dict['Type '+str(i)]['Instance '+str(j)]['c,A,b'] = np.concatenate((ins_lcl_v, ins_lcl_w, ins_lcl_b))

features = [*range(nr_ftres)] 

i 0
j 0
indc 0
1.751522533495737
j 1
indc 1
1.8502720988380643
j 2
indc 2
2.201448225923244
j 3
indc 3
2.281406295969403
j 4
indc 4
1.7669134102255122
j 5
indc 5
2.05015936923335
j 6
indc 6
1.858186648953836
j 7
indc 7
2.281883380106018
j 8
indc 8
2.1108784706417842
j 9
indc 9
2.030991555671629
i 1
j 0
indc 1
1.8502720988380643
j 1
indc 2
2.201448225923244
j 2
indc 3
2.281406295969403
j 3
indc 4
1.7669134102255122
j 4
indc 5
2.05015936923335
j 5
indc 6
1.858186648953836
j 6
indc 7
2.281883380106018
j 7
indc 8
2.1108784706417842
j 8
indc 9
2.030991555671629
j 9
indc 10
2.159561766978018
i 2
j 0
indc 2
2.201448225923244
j 1
indc 3
2.281406295969403
j 2
indc 4
1.7669134102255122
j 3
indc 5
2.05015936923335
j 4
indc 6
1.858186648953836
j 5
indc 7
2.281883380106018
j 6
indc 8
2.1108784706417842
j 7
indc 9
2.030991555671629
j 8
indc 10
2.159561766978018
j 9
indc 11
2.0234037140676673
i 3
j 0
indc 3
2.281406295969403
j 1
indc 4
1.7669134102255122
j 2
indc 5
2.05015936923335
j 3
indc 6
1.85818

In [6]:
# Help functions

# Create samples
def sample_perturbations_normal(orig, ftr_index_list, model_lcl, hyperprm = {}, mean = 0, var = 0.2, size = 1000, feasibility_check = True, bounded_check = True):
    
    org_plus_prtb = [orig]
    cntr = 1
    incr = 1

    while cntr < size:
        orig_with_noise = copy.deepcopy(orig)
        good_sample = True
        
        for j in range(len(orig)):
            if j in ftr_index_list:
                lcl_var = orig_with_noise[j] * var
                orig_with_noise[j] = orig_with_noise[j] + np.random.normal(mean, lcl_var)

        if feasibility_check:
            good_sample = good_sample * model_lcl(orig_with_noise, output = 'feasibility', **hyperprm)
        if bounded_check:
            good_sample = good_sample * model_lcl(orig_with_noise, output = 'bounded', **hyperprm)

        if good_sample == True:
            org_plus_prtb.append(np.asarray(orig_with_noise))
            cntr  = cntr + 1
        
        incr = incr + 1
        
    org_plus_prtb = np.asarray(org_plus_prtb)

    return org_plus_prtb

# Get values from samples
def get_values_from_samples(smpls, model_lcl, hyperprm = {}):
    values = []
    
    for smpl in smpls:
        values.append(model_lcl(smpl, **hyperprm))
    
    return values

# Determine weight of samples
def std_weight_function(a, b, ftr_index_list, kernel_width = None):
    d = np.linalg.norm(a - b)
    if kernel_width is None:
        krnl_wdth = 0.75 * len(ftr_index_list)
    else:
        krnl_wdth = kernel_width
    return np.exp(-(d ** 2) / (2* krnl_wdth ** 2))
    
def get_weights_from_samples(smpls, ftr_index_list, function = None, width = None):
    
    org = smpls[0]
    weights = []

    for smpl in smpls:
        if function is not None:
            weights.append(function(org, smpl))
        else:
            weights.append(std_weight_function(org, smpl, ftr_index_list, width))

    return weights

def get_knn(weights, k):
    return sorted(range(len(weights)), key=lambda i: weights[i])[-k:]

In [8]:
for ky1 in ins_dict.keys():
    for ky2 in ins_dict[ky1].keys():
        print(ky1,ky2)
        for i in range(rsmpl_sz):
            np.random.seed(seed_lst[i])
            b = 'Batch '+ str(i)
            ins_dict[ky1][ky2][b] = {}
            # Sample new instances
            samples = sample_perturbations_normal(ins_dict[ky1][ky2]['c,A,b'], features, model_lcl = model_KS_cts, size=smple_sz)
            ins_dict[ky1][ky2][b]['smpl_vls'] = samples[:,features]
            ins_dict[ky1][ky2][b]['Samples']  = np.concatenate((ins_dict[ky1][ky2][b]['smpl_vls'],np.ones((smple_sz,1))), axis = 1) 

            # Find output of samples
            ins_dict[ky1][ky2][b]['Actuals']  = get_values_from_samples(samples, model_KS_cts, hyperprm = {'output':'all'})

            # Find weights of samples
            d_list = []
            for x in samples:
                d_list.append(np.linalg.norm(samples[0] - x))

            ins_dict[ky1][ky2][b]['Weights'] = get_weights_from_samples(samples, features, width=np.mean(d_list))

Type 0 Instance 0
Type 0 Instance 1
Type 0 Instance 2
Type 0 Instance 3
Type 0 Instance 4
Type 0 Instance 5
Type 0 Instance 6
Type 0 Instance 7
Type 0 Instance 8
Type 0 Instance 9
Type 1 Instance 0
Type 1 Instance 1
Type 1 Instance 2
Type 1 Instance 3
Type 1 Instance 4
Type 1 Instance 5
Type 1 Instance 6
Type 1 Instance 7
Type 1 Instance 8
Type 1 Instance 9
Type 2 Instance 0
Type 2 Instance 1
Type 2 Instance 2
Type 2 Instance 3
Type 2 Instance 4
Type 2 Instance 5
Type 2 Instance 6
Type 2 Instance 7
Type 2 Instance 8
Type 2 Instance 9
Type 3 Instance 0
Type 3 Instance 1
Type 3 Instance 2
Type 3 Instance 3
Type 3 Instance 4
Type 3 Instance 5
Type 3 Instance 6
Type 3 Instance 7
Type 3 Instance 8
Type 3 Instance 9


In [45]:
import re

def parse_array(s):
    # Remove brackets, replace whitespace separators with commas
    s = s.strip().strip("[]")
    numbers = re.split(r"\s+", s.strip())
    return np.array([float(x) for x in numbers if x])

df_exp = pd.read_excel('Overview_Original_df_25_items2024-12-23.xlsx')
df_exp["Beta"] = df_exp["Beta"].apply(parse_array)

ValueError: could not convert string to float: '...'

In [ ]:
df_exp.types()

,Unnamed: 0,Type nr,Instance nr,Batch nr,Method,Beta,Error Prediction,Error Objective,Error Constraint,Contribution Target 0 Val\n1,...,Contribution Target 25 Wgt\n16,Contribution Target 25 Wgt\n17,Contribution Target 25 Wgt\n18,Contribution Target 25 Wgt\n19,Contribution Target 25 Wgt\n20,Contribution Target 25 Wgt\n21,Contribution Target 25 Wgt\n22,Contribution Target 25 Wgt\n23,Contribution Target 25 Wgt\n24,Contribution Target 25 Wgt\n25
0,0,Type 0,Instance 0,Batch 0,Standard Linear Regression,[[ 7.27711756e-01 9.16822078e-01 9.30962395e...,641.426558,0.324159,4.954856,0.083221,...,-0.039456,0.047251,-0.107175,-0.032991,-0.058021,-0.042201,0.002343,-0.022526,-0.021313,-1.221870
1,1,Type 0,Instance 0,Batch 0,Decision Tree Regression,[[ 7.27711756e-01 9.16822078e-01 9.30962395e...,546.482868,7.836525,21.637352,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.447098
2,2,Type 0,Instance 0,Batch 0,Regularized Linear Regression,[[ 0.70447027 0.97105556 0.89372081 ... ...,705.536172,0.096180,0.009867,0.080563,...,-0.011708,0.045005,-0.085047,-0.010506,-0.040612,-0.012731,-0.009108,-0.000831,-0.017112,-1.091296
3,3,Type 0,Instance 0,Batch 1,Standard Linear Regression,[[ 0.76329547 1.02601879 0.99274442 ... ...,645.520956,0.337653,5.051188,0.087290,...,0.008684,-0.007687,-0.057737,-0.008522,0.030135,-0.114816,0.008336,-0.033735,0.040701,-1.151329
4,4,Type 0,Instance 0,Batch 1,Decision Tree Regression,[[ 0.76329547 1.02601879 0.99274442 ... ...,549.845438,8.105729,22.334798,0.044649,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.493076
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,1195,Type 3,Instance 9,Batch 8,Decision Tree Regression,[[ 0.56692258 0.61002294 0.16783219 ... ...,857.492473,15.820764,41.073233,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000131,0.000000,0.000000,0.752132
1196,1196,Type 3,Instance 9,Batch 8,Regularized Linear Regression,[[ 6.13048728e-01 5.47010407e-01 -5.21054262e...,1177.523599,0.295839,0.031152,0.042740,...,-0.021023,-0.026089,0.012368,0.000362,-0.031884,0.016507,0.031982,-0.010359,0.006051,-0.486177
1197,1197,Type 3,Instance 9,Batch 9,Standard Linear Regression,[[ 0.48870906 0.40130994 0.31479861 ... ...,1066.515198,0.824887,9.328169,0.034071,...,0.016593,0.037616,0.005553,-0.031840,-0.003191,0.023267,0.018372,0.022494,0.025847,-0.468137
1198,1198,Type 3,Instance 9,Batch 9,Decision Tree Regression,[[ 0.48870906 0.40130994 0.31479861 ... ...,890.771028,15.669282,40.601983,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.822807


In [35]:
df_exp[(df_exp['Type nr'] == ky1) & (df_exp['Instance nr'] == ky2) & (df_exp['Batch nr'] == b) & (df_exp['Method'] == 'Standard Linear Regression')]

,Unnamed: 0,Type nr,Instance nr,Batch nr,Method,Beta,Error Prediction,Error Objective,Error Constraint,Contribution Target 0 Val\n1,...,Contribution Target 25 Wgt\n16,Contribution Target 25 Wgt\n17,Contribution Target 25 Wgt\n18,Contribution Target 25 Wgt\n19,Contribution Target 25 Wgt\n20,Contribution Target 25 Wgt\n21,Contribution Target 25 Wgt\n22,Contribution Target 25 Wgt\n23,Contribution Target 25 Wgt\n24,Contribution Target 25 Wgt\n25
1197,1197,Type 3,Instance 9,Batch 9,Standard Linear Regression,[[ 0.48870906 0.40130994 0.31479861 ... ...,1066.515198,0.824887,9.328169,0.034071,...,0.016593,0.037616,0.005553,-0.03184,-0.003191,0.023267,0.018372,0.022494,0.025847,-0.468137


In [29]:
df_explainers[df_explainers['Type nr'] == 'Type 0']['Beta'][0]

'[[ 7.27711756e-01  9.16822078e-01  9.30962395e-01 ... -6.47361337e-01\n  -5.41476847e-01  1.00036003e+00]\n [ 1.00691039e+01 -1.02733932e-01 -5.79062289e-01 ...  1.20477768e+00\n   7.02483316e-01  1.45580136e+00]\n [-1.82456052e-02  1.53260128e+00 -3.46558220e-01 ...  7.12928774e-02\n   2.71051337e-01  9.46973743e-01]\n ...\n [-6.94508790e-02  1.07417776e-01 -6.56706561e-01 ...  3.04265115e-01\n   2.85189536e-01  8.69226318e-01]\n [-1.32666991e+00 -3.88429192e-01  1.04508945e-02 ... -1.19924026e+01\n   5.81494222e-01  1.94579069e+00]\n [-1.42105242e+00 -2.82136507e-01 -1.03372504e+00 ... -2.39701904e-01\n  -1.89274585e+01  1.71889873e+00]]'

In [36]:
for ky1 in ins_dict.keys():
    for ky2 in ins_dict[ky1].keys():
        print(ky1,ky2)
        for i in range(rsmpl_sz):
            b = 'Batch '+ str(i)
            for method in ['Standard Linear Regression', 'Regularized Linear Regression']:
                df_lcl = df_exp[(df_exp['Type nr'] == ky1) & (df_exp['Instance nr'] == ky2) & (df_exp['Batch nr'] == b) & (df_exp['Method'] == method)]
                ins_dict[ky1][ky2][b][method] = np.array(df_lcl['Beta'])

Type 0 Instance 0
Type 0 Instance 1
Type 0 Instance 2
Type 0 Instance 3
Type 0 Instance 4
Type 0 Instance 5
Type 0 Instance 6
Type 0 Instance 7
Type 0 Instance 8
Type 0 Instance 9
Type 1 Instance 0
Type 1 Instance 1
Type 1 Instance 2
Type 1 Instance 3
Type 1 Instance 4
Type 1 Instance 5
Type 1 Instance 6
Type 1 Instance 7
Type 1 Instance 8
Type 1 Instance 9
Type 2 Instance 0
Type 2 Instance 1
Type 2 Instance 2
Type 2 Instance 3
Type 2 Instance 4
Type 2 Instance 5
Type 2 Instance 6
Type 2 Instance 7
Type 2 Instance 8
Type 2 Instance 9
Type 3 Instance 0
Type 3 Instance 1
Type 3 Instance 2
Type 3 Instance 3
Type 3 Instance 4
Type 3 Instance 5
Type 3 Instance 6
Type 3 Instance 7
Type 3 Instance 8
Type 3 Instance 9


In [40]:
ins_dict[ky1][ky2][b][method]

array(['[[  0.4962851    0.44704547   0.88786208 ...  -0.41674116  -0.2863792\n    0.92060566]\n [ 15.14838629  -2.54783397  -3.7107449  ...   0.17585512  -2.69329782\n    1.69355264]\n [ -0.27418976  19.74181144   0.5745459  ...   0.31306936   1.88924968\n    1.33845454]\n ...\n [ -0.03085487  -2.19749506   3.24073084 ...   0.44993674  -0.50626535\n    1.01888418]\n [ -1.59507892   0.14234371   1.92409681 ...  -7.87907546  -1.05142272\n    0.76750446]\n [  0.41182843  -0.59556444   2.20033506 ...   0.30867932 -15.83595153\n    0.25345944]]'],
      dtype=object)